# Classical Baseline Models

the scores every circuit is compared against. each model trains on all of ff++ train and is scored on val, with a bootstrap interval over videos. every model gets 3 settings picked on val, the same budget as the circuits.

In [1]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

seed = 0
np.random.seed(seed)

prep_dir = "/kaggle/input/datasets/malakkhaled202300369/nic-feature-dataset/prep"
clip_dir = "/kaggle/input/datasets/malakkhaled202300369/nic-clip-dataset/features"
out_dir = "/kaggle/working/baselines"
os.makedirs(out_dir, exist_ok=True)

c_grid = [0.01, 0.1, 1.0]          # logistic regression strength
mlp_hidden = 6                     # 8*6 + 6 + 6 + 1 = 61 weights, close to a depth-2 rot circuit (57)
alpha_grid = [1e-4, 1e-3, 1e-2]    # mlp weight decay

In [2]:
angles, meta = {}, {}
for split in ["train", "val"]:
    angles[split] = np.load(f"{prep_dir}/ffpp_{split}_angles32.npy").astype(np.float64)
    meta[split] = pd.read_csv(f"{prep_dir}/ffpp_{split}_rows.csv")

is_inner = np.load(f"{prep_dir}/ffpp_train_is_inner.npy").astype(bool)
comp = pd.read_csv(f"{prep_dir}/per_component_auc.csv")
best8 = sorted(comp.sort_values("strength_search_train", ascending=False)["component"].head(8).tolist())
first8 = list(range(8))

y_train = meta["train"]["label"].to_numpy()
y_val = meta["val"]["label"].to_numpy()
# a video is (method, video_id), the 4 fakes of one pair share a video_id
val_video = meta["val"].groupby(["method", "video_id"]).ngroup().to_numpy()

print("train frames:", len(y_train), " val frames:", len(y_val), " val videos:", val_video.max() + 1)
print("best 8:", best8)

train frames: 46076  val frames: 22396  val videos: 700
best 8: [2, 3, 4, 8, 10, 16, 17, 21]


In [3]:
clip = {}
for split in ["train", "val"]:
    clip_rows = pd.read_csv(f"{clip_dir}/ffpp_{split}_rows.csv")
    assert (clip_rows["path"].to_numpy() == meta[split]["path"].to_numpy()).all(), split
    x = np.load(f"{clip_dir}/ffpp_{split}_clip.npy").astype(np.float64)
    clip[split] = x / np.linalg.norm(x, axis=1, keepdims=True)

def video_level(prob):
    df = pd.DataFrame({"video": val_video, "p": prob, "y": y_val})
    g = df.groupby("video").agg(p=("p", "mean"), y=("y", "first"))
    return g["p"].to_numpy(), g["y"].to_numpy()


def bootstrap_ci(p, y, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    aucs = []
    while len(aucs) < n:
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() == y[idx].max():
            continue
        aucs.append(roc_auc_score(y[idx], p[idx]))
    return np.percentile(aucs, [2.5, 97.5])


def score(prob):
    pv, yv = video_level(prob)
    lo, hi = bootstrap_ci(pv, yv)
    return {"frame auc": roc_auc_score(y_val, prob), "video auc": roc_auc_score(yv, pv), "ci low": lo, "ci high": hi}


def sincos(a):
    return np.concatenate([np.sin(a), np.cos(a)], axis=1)


def logreg(c):
    return make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=5000))


def mlp(alpha):
    return make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(mlp_hidden,), alpha=alpha,
                                                         max_iter=300, early_stopping=True, random_state=seed))


table, probs = [], {}


def tune(name, make_model, grid, xt, xv, weights):
    # one fit per setting, keep the best val video auc
    best = None
    for setting in grid:
        model = make_model(setting)
        model.fit(xt, y_train)
        prob = model.predict_proba(xv)[:, 1]
        pv, yv = video_level(prob)
        auc = roc_auc_score(yv, pv)
        if best is None or auc > best[1]:
            best = (setting, auc, prob)
    probs[name] = best[2]
    row = {"model": name, "weights": weights, "setting": str(best[0])}
    row.update(score(best[2]))
    table.append(row)
    print(f"{name:40s} video auc {row['video auc']:.3f} ({row['ci low']:.3f} to {row['ci high']:.3f})  setting {best[0]}")

## models on 8 angles

the same three models on the first 8 and the best 8. the sin/cos model is the twin of a circuit with no entanglers (17 weights), the mlp has 61.

In [4]:
mlp_weights = 8 * mlp_hidden + 2 * mlp_hidden + 1
for label, cols in [("first 8", first8), ("best 8", best8)]:
    at, av = angles["train"][:, cols], angles["val"][:, cols]
    tune(f"logistic regression, {label}", logreg, c_grid, at, av, 9)
    tune(f"sin/cos logistic regression, {label}", logreg, c_grid, sincos(at), sincos(av), 17)
    tune(f"mlp 6 hidden, {label}", mlp, alpha_grid, at, av, mlp_weights)

logistic regression, first 8             video auc 0.636 (0.585 to 0.682)  setting 0.01
sin/cos logistic regression, first 8     video auc 0.645 (0.595 to 0.693)  setting 0.01
mlp 6 hidden, first 8                    video auc 0.646 (0.595 to 0.694)  setting 0.01
logistic regression, best 8              video auc 0.713 (0.669 to 0.753)  setting 0.01
sin/cos logistic regression, best 8      video auc 0.713 (0.667 to 0.754)  setting 0.01
mlp 6 hidden, best 8                     video auc 0.714 (0.671 to 0.753)  setting 0.0001


## greedy sin/cos selector

adds one component at a time, the one that most raises the inner fitness auc (sin/cos model trained on search-train, C = 1). val is never used for picking, same rule as the search.

In [5]:
st = ~is_inner
a = angles["train"]
selected = []
for step in range(8):
    best = None
    for j in range(32):
        if j in selected:
            continue
        cols = selected + [j]
        model = logreg(1.0).fit(sincos(a[st][:, cols]), y_train[st])
        auc = roc_auc_score(y_train[is_inner], model.predict_proba(sincos(a[is_inner][:, cols]))[:, 1])
        if best is None or auc > best[1]:
            best = (j, auc)
    selected.append(best[0])
    print(f"step {step + 1}: add {best[0]:2d}, inner auc {best[1]:.3f}")

greedy8 = sorted(selected)
with open(f"{out_dir}/greedy_selection.json", "w") as f:
    json.dump({"selected_sorted": greedy8, "order_added": [int(j) for j in selected]}, f, indent=1)
print("greedy 8:", greedy8, "| shared with best 8:", len(set(greedy8) & set(best8)))

at, av = angles["train"][:, greedy8], angles["val"][:, greedy8]
tune("sin/cos logistic regression, greedy 8", logreg, c_grid, sincos(at), sincos(av), 17)
tune("mlp 6 hidden, greedy 8", mlp, alpha_grid, at, av, mlp_weights)

step 1: add  8, inner auc 0.625
step 2: add 17, inner auc 0.649
step 3: add 21, inner auc 0.667
step 4: add  3, inner auc 0.684
step 5: add  9, inner auc 0.694
step 6: add 10, inner auc 0.703
step 7: add 16, inner auc 0.710
step 8: add  4, inner auc 0.717
greedy 8: [3, 4, 8, 9, 10, 16, 17, 21] | shared with best 8: 7
sin/cos logistic regression, greedy 8    video auc 0.716 (0.669 to 0.757)  setting 0.01
mlp 6 hidden, greedy 8                   video auc 0.711 (0.667 to 0.753)  setting 0.001


## larger inputs

all 32 components, 8 pls components from the 512 clip features (fitted on all of train here), and the linear probe on the 512.

In [6]:
tune("logistic regression, all 32", logreg, c_grid, angles["train"], angles["val"], 33)

pls = PLSRegression(n_components=8, scale=False)
pls.fit(clip["train"], y_train)
tune("pls 8 + logistic regression", logreg, c_grid, pls.transform(clip["train"]), pls.transform(clip["val"]), 9)

tune("linear probe, 512 clip", logreg, c_grid, clip["train"], clip["val"], 513)

logistic regression, all 32              video auc 0.756 (0.716 to 0.794)  setting 1.0
pls 8 + logistic regression              video auc 0.841 (0.808 to 0.871)  setting 0.1
linear probe, 512 clip                   video auc 0.893 (0.866 to 0.916)  setting 1.0


## results

weights counts trained weights only. the 3 settings were picked on val, so these are slightly optimistic.

In [7]:
table = pd.DataFrame(table)
table.to_csv(f"{out_dir}/baselines.csv", index=False)
print(table.round(3).to_string(index=False))

video_method = meta["val"].groupby(["method", "video_id"])["method"].first().to_numpy()
per_method = []
for name in probs:
    pv, yv = video_level(probs[name])
    for m in ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]:
        keep = (video_method == "original") | (video_method == m)
        per_method.append({"model": name, "method": m, "video auc": roc_auc_score(yv[keep], pv[keep])})

per_method = pd.DataFrame(per_method)
per_method.to_csv(f"{out_dir}/per_method_auc.csv", index=False)
print(per_method.pivot(index="model", columns="method", values="video auc").round(3).to_string())

                                model  weights setting  frame auc  video auc  ci low  ci high
         logistic regression, first 8        9    0.01      0.612      0.636   0.585    0.682
 sin/cos logistic regression, first 8       17    0.01      0.624      0.645   0.595    0.693
                mlp 6 hidden, first 8       61    0.01      0.623      0.646   0.595    0.694
          logistic regression, best 8        9    0.01      0.692      0.713   0.669    0.753
  sin/cos logistic regression, best 8       17    0.01      0.691      0.713   0.667    0.754
                 mlp 6 hidden, best 8       61  0.0001      0.690      0.714   0.671    0.753
sin/cos logistic regression, greedy 8       17    0.01      0.695      0.716   0.669    0.757
               mlp 6 hidden, greedy 8       61   0.001      0.685      0.711   0.667    0.753
          logistic regression, all 32       33     1.0      0.731      0.756   0.716    0.794
          pls 8 + logistic regression        9     0.1      